In [13]:
import pandas as pd

drivers = pd.DataFrame({"driver_id": [1, 2, 3, 4], "city": ["SF", "SF", "LA", "LA"]})

invitations = pd.DataFrame({
	"invite_id": [101, 102, 103, 104, 105, 106, 107, 108],
	"request_id": [201, 201, 202, 203, 204, 205, 206, 207],
	"driver_id": [1, 2, 1, 2, 3, 3, 1, 2],
	"sent_at": [
		    "2026-11-01 09:00", "2026-11-01 09:01",
        "2026-11-01 10:00", "2026-11-01 11:00",
        "2026-11-01 09:00", "2026-11-02 09:00",
        "2026-11-02 10:00", "2026-11-02 11:00"
    ],
    "accepted": [0, 1, 1, 0, 1, 0, 1, 1]
})


trips = pd.DataFrame({
	"request_id": [201, 202, 204, 206, 207],
	"driver_id": [2, 1, 3, 1, 2],
	"status": ["completed", "cancelled", "completed", "completed", "completed"],
	"pickup_minutes": ["5", None, "8", "4", "6"]
	})

print(drivers)
print(invitations)
print(trips)

   driver_id city
0          1   SF
1          2   SF
2          3   LA
3          4   LA
   invite_id  request_id  driver_id           sent_at  accepted
0        101         201          1  2026-11-01 09:00         0
1        102         201          2  2026-11-01 09:01         1
2        103         202          1  2026-11-01 10:00         1
3        104         203          2  2026-11-01 11:00         0
4        105         204          3  2026-11-01 09:00         1
5        106         205          3  2026-11-02 09:00         0
6        107         206          1  2026-11-02 10:00         1
7        108         207          2  2026-11-02 11:00         1
   request_id  driver_id     status pickup_minutes
0         201          2  completed              5
1         202          1  cancelled           None
2         204          3  completed              8
3         206          1  completed              4
4         207          2  completed              6


In [8]:
# 将 sent_at 转成 datetime
invitations["sent_at"] = pd.to_datetime(invitations["sent_at"])
print(invitations["sent_at"])

0   2026-11-01 09:00:00
1   2026-11-01 09:01:00
2   2026-11-01 10:00:00
3   2026-11-01 11:00:00
4   2026-11-01 09:00:00
5   2026-11-02 09:00:00
6   2026-11-02 10:00:00
7   2026-11-02 11:00:00
Name: sent_at, dtype: datetime64[ns]


In [11]:
# 将 pickup_minutes 转成数值
trips["pickup_minutes"] = pd.to_numeric(trips["pickup_minutes"])
print(trips["pickup_minutes"])

0    5.0
1    NaN
2    8.0
3    4.0
4    6.0
Name: pickup_minutes, dtype: float64


In [14]:
# 筛选已接受的邀请
accepted_invitations = invitations.loc[invitations["accepted"] == 1]
print(accepted_invitations)

   invite_id  request_id  driver_id           sent_at  accepted
1        102         201          2  2026-11-01 09:01         1
2        103         202          1  2026-11-01 10:00         1
4        105         204          3  2026-11-01 09:00         1
6        107         206          1  2026-11-02 10:00         1
7        108         207          2  2026-11-02 11:00         1


In [20]:
# 确保 sent_at 是 datetime 类型，然后筛选 11 月 2 日、司机 1 的邀请
selected_invitations = invitations.loc[(invitations["sent_at"].dt.date == pd.to_datetime('2026-11-02').date()) & (invitations["driver_id"] == 1)]
print(selected_invitations)

   invite_id  request_id  driver_id             sent_at  accepted
6        107         206          1 2026-11-02 10:00:00         1


In [31]:
# 按司机和时间排序，找每个司机最新的一次邀请
sorted_invitations = invitations.sort_values(by=["driver_id", "sent_at"])
latest_invitations = sorted_invitations.drop_duplicates(subset=["driver_id"], keep="last")
print(latest_invitations)

   invite_id  request_id  driver_id             sent_at  accepted
6        107         206          1 2026-11-02 10:00:00         1
7        108         207          2 2026-11-02 11:00:00         1
5        106         205          3 2026-11-02 09:00:00         0


In [61]:
# 计算每个司机的：邀请次数；接受次数；接单率, 并保留没有邀请的司机
# invite_counts = invitations["driver_id"].value_counts()
# accepted_invitations = invitations.loc[invitations["accepted"] == 1]
# accepted_counts = accepted_invitations["driver_id"].value_counts()
# accepted_rates = accepted_counts / invite_counts
# accepted_rates = accepted_rate.fillna(0)

#driver_stats = pd.DataFrame({"invite_count": invite_counts, "accepted_count": accepted_counts, "accepted_rate": accepted_rates})

driver_stats = invitations.groupby("driver_id", as_index=False).agg(invite_count = ("invite_id", "size"), accepted_count = ("accepted", "sum"))
#driver_stats["accepted_rate"] = driver_stats["accepted_count"] / driver_stats["invite_count"]

driver_stats = drivers[["driver_id"]].merge(driver_stats, on="driver_id", how="left").fillna(0)

print(driver_stats)

count_cols = ["invite_count", "accepted_count"]
driver_stats[count_cols] = driver_stats[count_cols].astype(int)

driver_stats["accepted_rate"] = driver_stats["accepted_count"] / driver_stats["invite_count"]


   driver_id  invite_count  accepted_count
0          1           3.0             2.0
1          2           3.0             2.0
2          3           2.0             1.0
3          4           0.0             0.0
